# RentSafe-MY — Scam Classifier Training

Fine-tunes `xlm-roberta-base` into the binary SCAM / LEGITIMATE classifier used by
`nlp-service` layer 2, then saves the weights for download.

This runs here rather than locally because AdamW needs ~4.4 GB of optimizer state for
this model, which does not fit in the dev machine's free RAM.

**Before running: Runtime → Change runtime type → T4 GPU.**

## 1. Confirm the GPU is attached

In [ ]:
import torch

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {total / 1e9:.1f} GB total, {free / 1e9:.1f} GB free")
else:
    print("No GPU attached.")
    print("Runtime > Change runtime type > T4 GPU, then re-run this cell.")
    print("(Training still works on CPU here, just far slower.)")

## 2. Install dependencies

Pinned to the same `transformers` version as the local service so the saved weights load without surprises.

In [ ]:
!pip install -q "transformers==4.46.2" "accelerate>=0.26.0" scikit-learn

## 3. Upload the dataset

Run the cell, then pick **both** `train.csv` and `val.csv` from **one** of:

- `rentsafe-my/data/processed/random/` — stratified split, in-distribution score
- `rentsafe-my/data/processed/grouped/` — whole templates held out, generalisation score

Run this notebook once per folder, and set `SPLIT_SET` in section 4 to match.
The held-out real conversations are never uploaded here — they are the final test
set and stay sealed until training is finished.

In [ ]:
import os
from google.colab import files

os.makedirs("data", exist_ok=True)
print("Select train.csv and val.csv from rentsafe-my/data/processed/ ...")

for name in files.upload():
    os.replace(name, os.path.join("data", name))

print("Uploaded:", sorted(os.listdir("data")))

## 4. Train

Mirrors `scripts_train.py`, but with AdamW and a larger batch size since memory is no longer the constraint.

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

MODEL = "xlm-roberta-base"
MAX_LENGTH = 192
CHECKPOINT_DIR = "checkpoints"

# Which split set you uploaded: "random" (in-distribution) or "grouped"
# (held-out templates). Keeps the two runs from overwriting each other.
SPLIT_SET = "random"
FINAL_DIR = f"xlm-roberta-scam-{SPLIT_SET}"
LABELS = {"LEGITIMATE": 0, "SCAM": 1}


def load_frame(path):
    frame = pd.read_csv(path)
    frame["label"] = frame["label"].str.upper().replace({"LEGIT": "LEGITIMATE"})
    invalid = sorted(set(frame["label"]) - set(LABELS))
    if invalid:
        raise ValueError(f"Unsupported labels in {path}: {invalid}")
    return frame


class ScamDataset(torch.utils.data.Dataset):
    def __init__(self, frame, tokenizer):
        self.encodings = tokenizer(
            frame["text"].tolist(), truncation=True, max_length=MAX_LENGTH
        )
        self.labels = [LABELS[label] for label in frame["label"]]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: value[idx] for key, value in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


def compute_metrics(prediction):
    predicted = np.argmax(prediction.predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        prediction.label_ids, predicted, average="binary", zero_division=0
    )
    return {
        "accuracy": accuracy_score(prediction.label_ids, predicted),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


tokenizer = AutoTokenizer.from_pretrained(MODEL)
train_dataset = ScamDataset(load_frame("data/train.csv"), tokenizer)
val_dataset = ScamDataset(load_frame("data/val.csv"), tokenizer)
print(f"{len(train_dataset)} train / {len(val_dataset)} validation examples")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    num_labels=2,
    id2label={0: "LEGITIMATE", 1: "SCAM"},
    label2id=LABELS,
)

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 5. Final scores

The FYP target is validation F1 >= 0.80.

In [ ]:
metrics = trainer.evaluate()

for key in ("eval_accuracy", "eval_precision", "eval_recall", "eval_f1"):
    print(f"{key.replace('eval_', ''):<10} {metrics[key]:.4f}")

print()
print("TARGET MET" if metrics["eval_f1"] >= 0.80 else "Below the 0.80 F1 target")

## 6. Save the best weights

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print(sorted(os.listdir(FINAL_DIR)))

## 7. Get the weights off Colab

Two options — Drive is more reliable for ~1.1 GB than a browser download.
Run **one** of the cells below.

In [ ]:
# Option A — copy to Google Drive
import shutil

from google.colab import drive

drive.mount("/content/drive")

destination = "/content/drive/MyDrive/rentsafe-my/xlm-roberta-scam"
os.makedirs(os.path.dirname(destination), exist_ok=True)
if os.path.exists(destination):
    shutil.rmtree(destination)
shutil.copytree(FINAL_DIR, destination)

print("Saved to", destination)

In [ ]:
# Option B — download a zip directly
import shutil

archive = shutil.make_archive(FINAL_DIR, "zip", FINAL_DIR)
print(f"{os.path.getsize(archive) / 1e6:.0f} MB")

files.download(archive)

## 8. Use the weights locally

Put the folder at `nlp-service/models/xlm-roberta-scam`, so it contains
`config.json`, `model.safetensors`, and the tokenizer files directly (not nested
inside another folder).

Then point the service at it in `nlp-service/.env`:

```
FINETUNED_MODEL_DIR=./models/xlm-roberta-scam
```

Inference needs only ~1.1 GB of RAM, so it runs fine on the local machine — it was
only training that did not fit.